**ADDESTRAMENTO ENCODER**

SETUP

In [1]:
import digitalhub as dh
import pandas as pd
import matplotlib.pyplot as plt

NOME_PROGETTO = "floods"
project = dh.get_project(NOME_PROGETTO)
print(f"Progetto: {project.name}")

Progetto: floods


TRAINING

In [2]:
encoders_train_func = project.new_function(
    name="encoders_train-job-v20",
    kind="python",
    python_version="PYTHON3_10",
    code_src="Encoders/", 
    handler="pretrain_encoders", 
    base_image="pytorch/pytorch:2.1.2-cuda11.8-cudnn8-runtime",
    requirements=["pandas==2.3.3", "numpy==2.2.6", "rasterio==1.4.4", "tqdm==4.70.0", "tifffile==2024.8.30"]
)

build = encoders_train_func.run("build", wait=True)
print(f"BUILD: {build.status.state}")



2026-08-06 12:49:35,733 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 9a6fd1984723447fbea420943b9790ac to finish...
2026-08-06 12:49:40,740 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 9a6fd1984723447fbea420943b9790ac to finish...
2026-08-06 12:49:45,750 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 9a6fd1984723447fbea420943b9790ac to finish...
2026-08-06 12:49:50,761 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 9a6fd1984723447fbea420943b9790ac to finish...
2026-08-06 12:49:55,769 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 9a6fd1984723447fbea420943b9790ac to finish...
2026-08-06 12:50:00,778 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 9a6fd1984723447fbea420943b9790ac to finish...
2026-08-06 12:50:05,787 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 9a6fd1984723447fbea420943b9790ac to finish...
2026-08-06 12

BUILD: COMPLETED


RISULTATI

In [3]:
# setup ambiente
volumi = [
    {
        "volume_type": "ephemeral",
        "name": "volume-spazio-dati",
        "mount_path": "/data",      
        "spec": {"size": "250Gi"}   
    }
]

parametri = {
    "epochs": 1, 
    "batch_size": 16, 
    "lr": 1e-4, 
    "weight_decay": 1e-4,      
    "patch_size": 256, 
    "n_images1": 4, "n_channels1": 2,                    # sar
    "n_images2": 4, "n_channels2": 10,                   # ottiche               
    "mamba": False, 
    "workers": 4
}

print(f"PARAMETRI: {parametri}")

run_train_encoders = encoders_train_func.run(
    action="job", 
    parameters=parametri, 
    volumes=volumi, 
    profile="1xV100",
    wait=True)

print(f"STATO FINALE: {run_train_encoders.status.state}")
print(run_train_encoders.logs())

2026-08-06 12:50:32,553 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run e8aa24139821410cb982192e4c4e2d1b to finish...


PARAMETRI: {'epochs': 1, 'batch_size': 16, 'lr': 0.0001, 'weight_decay': 0.0001, 'patch_size': 256, 'n_images1': 4, 'n_channels1': 2, 'n_images2': 4, 'n_channels2': 10, 'mamba': False, 'workers': 4}


2026-08-06 12:50:37,560 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run e8aa24139821410cb982192e4c4e2d1b to finish...
2026-08-06 12:50:42,674 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run e8aa24139821410cb982192e4c4e2d1b to finish...
2026-08-06 12:50:47,686 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run e8aa24139821410cb982192e4c4e2d1b to finish...
2026-08-06 12:50:52,800 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run e8aa24139821410cb982192e4c4e2d1b to finish...
2026-08-06 12:50:57,921 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run e8aa24139821410cb982192e4c4e2d1b to finish...
2026-08-06 12:51:02,930 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run e8aa24139821410cb982192e4c4e2d1b to finish...
2026-08-06 12:51:07,940 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run e8aa24139821410cb982192e4c4e2d1b to finish...
2026-08-06 12

In [ ]:
# salvataggio log
print("SALVATAGGIO METRICHE")
path_s1 = project.get_artifact("metrics-s1").download()
path_s2 = project.get_artifact("metrics-s2").download()

# risultati grafici
df_s1 = pd.read_csv(path_s1)
df_s2 = pd.read_csv(path_s2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# sar
ax1.plot(df_s1['epoch'], df_s1['train_loss'], color='blue', label='Train Loss')
ax1.set_title('Pre-training SAR')
ax1.set_xlabel('Epoche')
ax1.set_ylabel('Loss (MSE)')
ax1.grid(True)

# ottico
ax2.plot(df_s2['epoch'], df_s2['train_loss'], color='red', label='Train Loss')
ax2.set_title('Pre-training OTTICO')
ax2.set_xlabel('Epoche')
ax2.grid(True)

plt.show()